In [ ]:
import kaggle
from pathlib import Path

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    "teejmahal20/airline-passenger-satisfaction",
    path=DATA_DIR,
    unzip=True,
)

print("Pobrane pliki:")
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
import os
import sys
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning:pkg_resources"

sys.path.append("..")

from src.preprocessing import (
    load_data,
    analyze_missing_values,
    clean_data,
    build_preprocessor,
    plot_correlation_matrix,
    plot_target_correlation,
    engineer_features,
    split_X_y,
)
from src.evaluation import evaluate, cross_validate_model, plot_pr_curve
from src.models.xgboost_model import tune_hyperparameters
from src.models.logistic_regression import tune_hyperparameters as tune_hyperparameters_lr
from src.models.random_forest import tune_hyperparameters as tune_hyperparameters_rf
from src.scripts.run_selectkbest_experiment import (
    run_selectkbest_experiment,
    get_feature_scores,
    plot_k_vs_metrics,
    plot_feature_scores,
)
from src.scripts.run_rfe_experiment import (
    run_rfe_experiment,
    get_feature_rankings,
    plot_n_features_vs_metrics,
    plot_feature_rankings,
)

In [ ]:
train, test = load_data()

print("=== Train ===")
print(train.shape)
train.head()

In [ ]:
print("=== Braki w train ===")
analyze_missing_values(train)

print("\n=== Braki w test ===")
analyze_missing_values(test)

In [ ]:
train_clean = clean_data(train)
test_clean = clean_data(test)

print("Train po czyszczeniu:", train_clean.shape)
print("Test po czyszczeniu: ", test_clean.shape)

In [ ]:
plot_correlation_matrix(train_clean)

In [ ]:
plot_target_correlation(train_clean)

In [ ]:
train_feat = engineer_features(train_clean)
test_feat = engineer_features(test_clean)

print("Nowe kolumny:", [c for c in train_feat.columns if c not in train_clean.columns])
train_feat[["Departure Delay in Minutes", "Arrival Delay in Minutes", "total_delay", "avg_service_score"]].describe()

In [ ]:
X_train, y_train = split_X_y(train_feat)
X_test, y_test = split_X_y(test_feat)

preprocessor = build_preprocessor(X_train)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("Rozkład klas y_train:\n", y_train.value_counts())

## Model Regresji Logistycznej

In [ ]:
search = tune_hyperparameters_lr(preprocessor, X_train, y_train)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train)

print("Walidacja krzyżowa najlepszego modelu (5-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="Regresja logistyczna")

## Model Las Losowy

In [ ]:
search = tune_hyperparameters_rf(preprocessor, X_train, y_train)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train)

print("Walidacja krzyżowa najlepszego modelu (5-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="Las losowy")

## Model XGBoost

In [ ]:
search = tune_hyperparameters(preprocessor, X_train, y_train)

print("Najlepsze parametry:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nNajlepsze F1 (CV): {search.best_score_:.4f}")

In [ ]:
cv_scores = cross_validate_model(search.best_estimator_, X_train, y_train)

print("Walidacja krzyżowa najlepszego modelu (5-fold):")
print(cv_scores.to_string(index=False))
print("\nŚrednie:")
print(cv_scores.mean().map("{:.4f}".format).to_string())
print("\nOdchylenia standardowe:")
print(cv_scores.std().map("{:.4f}".format).to_string())

In [ ]:
best_model = search.best_estimator_
metrics = evaluate(best_model, X_test, y_test)

In [ ]:
plot_pr_curve(best_model, X_test, y_test, label="XGBoost")

## Eksperyment: TabPFN vs pozostałe modele dla różnych rozmiarów próbek treningowych

Porównanie jakości obu modeli na tym samym zbiorze testowym, przy różnej liczbie próbek użytych do trenowania. Celem jest porównanie wyników modeli dla małych próbek danych.

In [ ]:
import subprocess, sys, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
script = "src/scripts/run_tabpfn_experiment.py"
with subprocess.Popen(
    [sys.executable, "-u", str(script)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd="..",
) as proc:
    for line in iter(proc.stdout.readline, ""):
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"\nBłąd: skrypt zakończył się kodem {proc.returncode}")

In [ ]:
results_df = pd.read_csv("../data/tabpfn_experiment_results.csv").set_index(["model", "sample_size"])
print(results_df.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, metric in zip(axes, ["accuracy", "f1", "roc_auc"]):
    for model_name, group in results_df.reset_index().groupby("model"):
        ax.plot(group["sample_size"], group[metric], marker="o", label=model_name)
    ax.set_title(metric)
    ax.set_xlabel("Liczba próbek treningowych")
    ax.set_ylabel(metric)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("TabPFN vs XGBoost - wpływ liczby próbek treningowych")
plt.tight_layout()
plt.show()

## Eksperyment: Wpływ liczby wybranych cech na jakość modelu przy użyciu algorytmu SelectKBest

Porównanie jakości modeli Linear Regression wytrenowanych na zbiorze treningowym z różną liczbą wybranych cech za pomocą algorytmu SelectKBest. Celem jest zbadanie, jak liczba cech wpływa na wyniki modeli.

In [ ]:
results = run_selectkbest_experiment(
    preprocessor, X_train, y_train, X_test, y_test,
    k_values=[5, 10, 15, 20, 25, "all"],
)

In [ ]:
plot_k_vs_metrics(results)

In [ ]:
scores_df = get_feature_scores(preprocessor, X_train, y_train)
scores_df

In [ ]:
plot_feature_scores(scores_df, top_n=29)

## Eksperyment: Wpływ liczby wybranych cech na jakość modelu przy użyciu algorytmu RFE

Porównanie jakości modelu Regresji Logistycznej w zależności od liczby wybranych cech (`n_features_to_select`) oraz estymatora użytego do rankowania cech (`estimator`). Badane estymatory: Regresja Logistyczna (LR) i Las Losowy (RF).

In [ ]:
results_rfe = run_rfe_experiment(preprocessor, X_train, y_train, X_test, y_test)

In [ ]:
print(results_rfe.to_string(index=False))

In [ ]:
plot_n_features_vs_metrics(results_rfe)

In [ ]:
rankings_lr = get_feature_rankings(preprocessor, X_train, y_train, estimator_name="LR")
rankings_lr

In [ ]:
plot_feature_rankings(rankings_lr, estimator_name="LR", top_n=29)

In [ ]:
rankings_rf = get_feature_rankings(preprocessor, X_train, y_train, estimator_name="RF")
rankings_rf

In [ ]:
plot_feature_rankings(rankings_rf, estimator_name="RF", top_n=29)